### RAG

In [37]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from gitdb.fun import chunk_size
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser



In [ ]:


pdf_path ="sbc-sample.pdf"

load_pdf = PyPDFLoader(pdf_path)
pages=load_pdf.load()

print(f"Loaded {len(pages)} pagesfrom the pdf.")
print("\n--- First page preview (first 500 charts)---")

Loaded 8 pagesfrom the pdf.

--- First page preview (first 500 charts)---


In [31]:
print(pages[0].page_content[:500])

1 of 8 
Insurance Company 1: Plan Option 1 Coverage Period: 01/01/2013 – 12/31/2013 
Summary of Benefits and Coverage: What this Plan Covers & What it Costs Coverage for: Individual + Spouse | Plan Type: PPO 
Questions: Call 1-800-[insert] or visit us at www.[insert]. 
If you aren’t clear about any of the underlined terms used in this form, see the Glossary.  You can view the Glossary 
at www.[insert] or call 1-800-[insert] to request a copy. 
 
 
This is only a summary. If you want more detail 


In [32]:
splitter= RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n","\n","."," "],
)
chunks=splitter.split_documents(pages)

In [33]:
chunks[0].page_content


'1 of 8 \nInsurance Company 1: Plan Option 1 Coverage Period: 01/01/2013 – 12/31/2013 \nSummary of Benefits and Coverage: What this Plan Covers & What it Costs Coverage for: Individual + Spouse | Plan Type: PPO \nQuestions: Call 1-800-[insert] or visit us at www.[insert]. \nIf you aren’t clear about any of the underlined terms used in this form, see the Glossary.  You can view the Glossary \nat www.[insert] or call 1-800-[insert] to request a copy. \n \n \nThis is only a summary. If you want more detail about your coverage and costs, you can get the complete terms in the policy or plan'

In [34]:

embedding = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
Vectore_Store = Chroma.from_documents(chunks,embedding)


print(f"Vector Store Ready. {Vectore_Store._collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1800.49it/s]


Vector Store Ready. 80


In [35]:
retriever = Vectore_Store.as_retriever(search_kwargs={"k":3})

text_query="What is the overall deductible for this plan?"
retrieved=retriever.invoke(text_query)

for i ,doc in enumerate(retrieved,1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:100])
    print()


--- Chunk 1 ---
document at www.[insert] or by calling 1-800-[insert]. 
  
Important Questions Answers Why this Matt

--- Chunk 2 ---
document at www.[insert] or by calling 1-800-[insert]. 
  
Important Questions Answers Why this Matt

--- Chunk 3 ---
Are there other 
deductibles for specific 
services? 
Yes. $300 for prescription drug 
coverage.  Th



In [42]:
def format_doc(docs):
    return "\n\n--\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """You are a health-insurance benefits assistant. Answer the question using ONLY the context provided below, which comes from a Summary of Benefits and Coverage (SBC) document.


Context:
{context}
"""


prompt = ChatPromptTemplate.from_messages([
    ("system",SYSTEM_PROMPT),
    ("human","{question}"),]
)
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)
chain =(
    {
        "context":retriever|format_doc,"question":RunnablePassthrough()
    }
    |prompt
    |llm
    |StrOutputParser()
)
print("RAG Chain Assembled.")



RAG Chain Assembled.


In [46]:
from transformers.models.tapas.tokenization_tapas import Question
question= "Are there separate deductibles for prescription drugs?"
print(f"Q: {question}\n")
print("A:",chain.invoke(question))

Q: Are there separate deductibles for prescription drugs?

A: Yes, there is a separate deductible for prescription drug coverage. The specific deductible is **$300** for prescription drugs. This means you must pay all costs for prescription drugs up to $300 before the plan begins to cover these services. There are no other specific deductibles for other services.
